# Demo: LangChain LCEL Variant (Parallel + Structured Outputs)

This notebook is a second LangChain variant that mirrors the planner-writer-editor flow using LCEL RunnableSequence, RunnableParallel, and Pydantic structured parsing.

## Preparing environment

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from time import perf_counter
from typing import List

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableSequence

from IPython.display import Markdown, display

In [3]:
TOPIC = "Artificial Intelligence"
VERBOSE = True

## Initialize LLM

In [4]:
llm = ChatOllama(
    model="mistral:latest",
    base_url="http://localhost:11434",
    temperature=0.7,
)

## Define Structured Schemas

In [5]:
class TrendReport(BaseModel):
    top_trends: List[str] = Field(description="Most important trends related to the topic")
    key_players: List[str] = Field(description="Key organizations or people driving the space")
    notable_news: List[str] = Field(description="Recent noteworthy developments")
    sources: List[str] = Field(description="Credible references or publication names")


class AudienceAnalysis(BaseModel):
    target_audience: str = Field(description="Primary audience description")
    interests: List[str] = Field(description="Audience interests")
    pain_points: List[str] = Field(description="Audience pain points")
    seo_keywords: List[str] = Field(description="SEO keywords relevant to the topic")


class ContentPlan(BaseModel):
    title: str = Field(description="Proposed blog title")
    outline: List[str] = Field(description="Section by section article outline")
    call_to_action: str = Field(description="Concise CTA for readers")
    audience_summary: str = Field(description="How this plan matches audience needs")
    references: List[str] = Field(description="References to use while writing")


class DraftArticle(BaseModel):
    markdown_article: str = Field(description="Draft article in markdown")


class EditedArticle(BaseModel):
    markdown_article: str = Field(description="Final polished markdown article")
    editor_notes: List[str] = Field(description="Short notes about edits applied")

## Build LCEL Chains

Two research branches run in parallel first, then outputs are merged into planning, writing, and editing chains in a RunnableSequence.

In [6]:
trend_parser = PydanticOutputParser(pydantic_object=TrendReport)
audience_parser = PydanticOutputParser(pydantic_object=AudienceAnalysis)
plan_parser = PydanticOutputParser(pydantic_object=ContentPlan)
draft_parser = PydanticOutputParser(pydantic_object=DraftArticle)
edit_parser = PydanticOutputParser(pydantic_object=EditedArticle)

trend_prompt = ChatPromptTemplate.from_template(
    """You are a research analyst. Analyze topic: {topic}.\n"
    "Focus on latest trends, key players, notable news, and references.\n\n"
    "{format_instructions}"""
)

audience_prompt = ChatPromptTemplate.from_template(
    """You are a market analyst. Analyze audience for topic: {topic}.\n"
    "Identify target audience, interests, pain points, and SEO keywords.\n\n"
    "{format_instructions}"""
)

plan_prompt = ChatPromptTemplate.from_template(
    """You are a content planner for topic: {topic}.\n"
    "Use these structured inputs to create a blog content plan.\n\n"
    "Trend report:\n{trend_report_json}\n\n"
    "Audience analysis:\n{audience_analysis_json}\n\n"
    "{format_instructions}"""
)

writer_prompt = ChatPromptTemplate.from_template(
    """You are a content writer for topic: {topic}.\n"
    "Write a compelling, accurate markdown article from this plan:\n{plan_json}\n\n"
    "The article should have an engaging intro, clearly titled sections, and a concise conclusion.\n"
    "{format_instructions}"""
)

editor_prompt = ChatPromptTemplate.from_template(
    """You are an editor. Improve grammar, clarity, and consistency for this markdown draft on {topic}.\n"
    "Return a polished final version and notes.\n\n"
    "Draft:\n{draft_markdown}\n\n"
    "{format_instructions}"""
)

trend_chain = trend_prompt | llm | trend_parser
audience_chain = audience_prompt | llm | audience_parser
plan_chain = plan_prompt | llm | plan_parser
writer_chain = writer_prompt | llm | draft_parser
editor_chain = editor_prompt | llm | edit_parser

parallel_research = RunnableParallel(
    trend=(
        RunnableLambda(lambda x: {
            "topic": x["topic"],
            "format_instructions": trend_parser.get_format_instructions(),
        })
        | trend_chain
    ),
    audience=(
        RunnableLambda(lambda x: {
            "topic": x["topic"],
            "format_instructions": audience_parser.get_format_instructions(),
        })
        | audience_chain
    ),
    topic=RunnableLambda(lambda x: x["topic"]),
)

pipeline = RunnableSequence(
    parallel_research,
    RunnableLambda(lambda x: {
        "topic": x["topic"],
        "trend": x["trend"],
        "audience": x["audience"],
        "trend_report_json": x["trend"].model_dump_json(indent=2),
        "audience_analysis_json": x["audience"].model_dump_json(indent=2),
    }),
    RunnableParallel(
        plan=(
            RunnableLambda(lambda x: {
                "topic": x["topic"],
                "trend_report_json": x["trend_report_json"],
                "audience_analysis_json": x["audience_analysis_json"],
                "format_instructions": plan_parser.get_format_instructions(),
            })
            | plan_chain
        ),
        context=RunnableLambda(lambda x: x),
    ),
    RunnableLambda(lambda x: {
        "topic": x["context"]["topic"],
        "trend": x["context"]["trend"],
        "audience": x["context"]["audience"],
        "plan": x["plan"],
        "plan_json": x["plan"].model_dump_json(indent=2),
    }),
    RunnableParallel(
        draft=(
            RunnableLambda(lambda x: {
                "topic": x["topic"],
                "plan_json": x["plan_json"],
                "format_instructions": draft_parser.get_format_instructions(),
            })
            | writer_chain
        ),
        context=RunnableLambda(lambda x: x),
    ),
    RunnableLambda(lambda x: {
        "topic": x["context"]["topic"],
        "trend": x["context"]["trend"],
        "audience": x["context"]["audience"],
        "plan": x["context"]["plan"],
        "draft": x["draft"],
    }),
    RunnableParallel(
        final=(
            RunnableLambda(lambda x: {
                "topic": x["topic"],
                "draft_markdown": x["draft"].markdown_article,
                "format_instructions": edit_parser.get_format_instructions(),
            })
            | editor_chain
        ),
        context=RunnableLambda(lambda x: x),
    ),
    RunnableLambda(lambda x: {
        "trend": x["context"]["trend"],
        "audience": x["context"]["audience"],
        "plan": x["context"]["plan"],
        "draft": x["context"]["draft"],
        "final": x["final"],
    }),
)

## Execute Pipeline

In [7]:
start = perf_counter()
outputs = pipeline.invoke({"topic": TOPIC})
end = perf_counter()

if VERBOSE:
    print(f"Trend items: {len(outputs['trend'].top_trends)}")
    print(f"Audience keywords: {len(outputs['audience'].seo_keywords)}")
    print(f"Plan sections: {len(outputs['plan'].outline)}")
    print(f"Draft length: {len(outputs['draft'].markdown_article)}")
    print(f"Final length: {len(outputs['final'].markdown_article)}")

print(f"\n{'=' * 100}\nLCEL pipeline finished in {end - start:.2f} seconds.\n{'=' * 100}\n")

Trend items: 4
Audience keywords: 11
Plan sections: 7
Draft length: 2281
Final length: 2281

LCEL pipeline finished in 28.65 seconds.



## Inspect Structured Outputs

In [8]:
print("=== Content Plan ===")
print(outputs['plan'].model_dump_json(indent=2))
print("\n=== Editor Notes ===")
for note in outputs['final'].editor_notes:
    print(f"- {note}")

=== Content Plan ===
{
  "title": "Trending AI Advancements and Their Impact on Businesses and Society",
  "outline": [
    "Introduction to AI Trends",
    "Large Language Models (LLMs) like ChatGPT: A Game Changer?",
    "Edge AI and IoT Integration: The Future of Decentralized Processing",
    "Autonomous Vehicles and Robotics: Navigating the Path Ahead",
    "AI Ethics and Regulation: Balancing Progress with Responsibility",
    "Notable News in AI",
    "Conclusion and Looking Forward"
  ],
  "call_to_action": "Stay updated on the latest AI trends and developments by following our blog.",
  "audience_summary": "This plan caters to professionals, students, researchers, data analysts, and business leaders who are interested in staying informed about AI technologies, machine learning algorithms, deep learning applications, robotics, automation, AI ethics, and regulations.",
  "references": [
    "Nature.com",
    "Wired",
    "VentureBeat",
    "TechCrunch"
  ]
}

=== Editor Notes ==

## Display Final Markdown

In [9]:
final_markdown = outputs['final'].markdown_article.strip()
if final_markdown.startswith("```"):
    lines = final_markdown.splitlines()
    if len(lines) > 2 and lines[-1].strip() == "```":
        final_markdown = "\n".join(lines[1:-1])
display(Markdown(final_markdown))

# Trending AI Advancements and Their Impact on Businesses and Society

![](ai-trends.jpg)

## Introduction to AI Trends
Artificial Intelligence (AI) is rapidly evolving, transforming industries and societies at an unprecedented pace. In this article, we will delve into some of the most exciting advancements in AI and explore their potential implications on businesses and society as a whole.

## Large Language Models (LLMs) like ChatGPT: A Game Changer?
[ChatGPT](https://www.wired.com/story/chatgpt-openai-launches-ai-powered-chatbot/) is making waves in the tech world by providing a conversational interface that can answer questions, write essays, summarize articles, and even generate code snippets. But what does this mean for businesses? Read on to find out more.

## Edge AI and IoT Integration: The Future of Decentralized Processing
[Edge AI](https://venturebeat.com/2021/05/19/edge-ai-will-drive-the-next-wave-of-innovation/) is poised to revolutionize the way we interact with IoT devices by enabling on-device processing and decision making. Learn how this advancement could change the future of decentralized computing.

## Autonomous Vehicles and Robotics: Navigating the Path Ahead
The advent of [autonomous vehicles](https://www.nature.com/articles/s41562-021-00897-y) and robotics is set to reshape transportation and manufacturing industries alike. In this section, we'll take a closer look at the current state of these technologies and discuss what lies ahead.

## AI Ethics and Regulation: Balancing Progress with Responsibility
As AI continues to advance, it becomes increasingly important for policymakers and stakeholders to address ethical concerns and establish regulations that promote responsible development. Discover some of the latest developments in this field and learn how they may impact your business.

## Notable News in AI
Stay up-to-date with the latest news and insights in the world of AI by following our blog: [Follow Our Blog](https://blog.yourwebsite.com)

## Conclusion and Looking Forward
In conclusion, artificial intelligence is poised to bring about significant changes across various industries. By staying informed about these trends and advancements, businesses can position themselves for success in the AI-driven future.